In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[2]:


class JupyterVolva:
    """Simplified interface for Jupyter notebooks"""
    
    def __init__(self, wavelength_name: str = "Cu Ka"):
        self.config = VolvaConfig()
        self.wavelength_name = wavelength_name
        self.wavelength = Constants.WAVELENGTHS[wavelength_name]['ka1']
        self.wavelength_info = Constants.WAVELENGTHS[wavelength_name]
        
        self.scherrer_analyzer = ScherrerAnalyzer(self.wavelength)
        self.monshi_analyzer = MonshiScherrerAnalyzer(self.wavelength_info)
        self.wh_analyzer = W#!/usr/bin/env python3
"""
Volva - XRD Analysis Tool (Jupyter Notebook Version)
Complete XRD crystallite size analysis with GUI launch capability
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from scipy.ndimage import minimum_filter1d
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext, ttk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
import logging
from datetime import datetime
import os
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

# Optional dependencies
try:
    from lmfit import models
    LMFIT_AVAILABLE = True
except ImportError:
    LMFIT_AVAILABLE = False
    print("Warning: lmfit not available. Using simplified peak fitting.")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION AND CONSTANTS
# =============================================================================

@dataclass
class VolvaConfig:
    """Configuration class for Volva application"""
    proximity_tolerance_deg: float = 5.0
    default_bg_filter_window: int = 151
    default_prominence_factor: float = 0.1
    default_min_width_deg: float = 0.05
    default_fitting_window_deg: float = 1.0
    default_scherrer_k: str = "Spherical (FWHM)"
    default_wh_k: float = 4.0
    default_wavelength: str = "Cu Ka"
    min_r_squared: float = 0.8
    default_bg_method: str = "polynomial"
    default_poly_order: int = 3
    default_peak_model: str = "voigt"

class Constants:
    """Physical constants and wavelength data"""
    WAVELENGTHS = {
        "Cu Ka": {"ka1": 0.15406, "ka2": 0.154439, "ratio": 0.5, "alpha": 2.2e-3},
        "Mo Ka": {"ka1": 0.07093, "ka2": 0.071073, "ratio": 0.5, "alpha": 1.86e-3},
        "Ag Ka": {"ka1": 0.05594, "ka2": 0.05638, "ratio": 0.5, "alpha": 1.54e-3},
        "Co Ka": {"ka1": 0.178897, "ka2": 0.179285, "ratio": 0.5, "alpha": 2.39e-3},
        "Cr Ka": {"ka1": 0.228970, "ka2": 0.229361, "ratio": 0.5, "alpha": 2.93e-3},
    }
    
    SCHERRER_CONSTANTS = {
        "General/Approximation": 0.9,
        "Spherical (FWHM)": 0.94,
        "Spherical (Integral Breadth)": 0.89,
        "Cubic": 1.0,
        "Rod-like": 1.84,
        "Plate-like": 0.89
    }

# =============================================================================
# DATA PROCESSING
# =============================================================================

class XRDDataProcessor:
    """XRD data loading and processing"""
    
    @staticmethod
    def load_xrd_file(file_path: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], str]:
        """Load XRD data with automatic format detection"""
        try:
            # Try different delimiters
            for delimiter in [None, '\t', ',', ' ']:
                try:
                    data = np.loadtxt(file_path, delimiter=delimiter)
                    if data.ndim == 2 and data.shape[1] >= 2:
                        two_theta = data[:, 0]
                        intensity = data[:, 1]
                        
                        # Validate data
                        if len(two_theta) < 10:
                            continue
                        if np.min(two_theta) < 0:
                            continue
                        
                        # Sort by 2theta
                        sort_indices = np.argsort(two_theta)
                        two_theta = two_theta[sort_indices]
                        intensity = intensity[sort_indices]
                        intensity = np.maximum(intensity, 0)  # Remove negative values
                        
                        return two_theta, intensity, f"Data loaded: {len(two_theta)} points"
                except:
                    continue
            
            return None, None, "Could not parse data file"
            
        except Exception as e:
            return None, None, f"Error loading file: {str(e)}"

    @staticmethod
    def subtract_background(two_theta: np.ndarray, intensity: np.ndarray, 
                          method: str = 'polynomial', poly_order: int = 3) -> Tuple[np.ndarray, np.ndarray]:
        """Background subtraction"""
        try:
            if method == 'polynomial':
                # Simple polynomial background
                coeffs = np.polyfit(two_theta, intensity, poly_order)
                background = np.polyval(coeffs, two_theta)
            else:  # rolling minimum
                window = min(151, len(intensity) // 5)
                if window % 2 == 0:
                    window += 1
                background = minimum_filter1d(intensity, size=window, mode='nearest')
            
            # Ensure background doesn't exceed intensity
            background = np.minimum(background, intensity * 0.95)
            background_subtracted = intensity - background
            background_subtracted = np.maximum(background_subtracted, 0)
            
            return background_subtracted, background
            
        except Exception as e:
            logger.warning(f"Background subtraction failed: {e}")
            return intensity, np.zeros_like(intensity)

# =============================================================================
# PEAK DETECTION AND FITTING
# =============================================================================

class PeakDetector:
    """Peak detection and fitting"""
    
    @staticmethod
    def detect_peaks(two_theta: np.ndarray, intensity: np.ndarray,
                    prominence_factor: float = 0.1, min_width_deg: float = 0.05) -> List[Dict]:
        """Detect peaks in XRD data"""
        try:
            # Smooth data
            if len(intensity) > 20:
                window_length = min(51, max(5, len(intensity)//20))
                if window_length % 2 == 0:
                    window_length += 1
                smoothed = savgol_filter(intensity, window_length, 3)
            else:
                smoothed = intensity.copy()
            
            # Calculate detection parameters
            step_size = np.median(np.diff(two_theta))
            min_width_points = max(3, int(min_width_deg / step_size))
            min_distance = max(5, int(0.1 / step_size))
            
            baseline = np.percentile(smoothed, 10)
            noise_level = np.std(smoothed[smoothed < np.percentile(smoothed, 30)])
            peak_height = baseline + max(3 * noise_level, prominence_factor * (np.max(smoothed) - baseline))
            
            # Find peaks
            peaks, properties = find_peaks(
                smoothed,
                height=peak_height,
                distance=min_distance,
                width=min_width_points,
                prominence=prominence_factor * np.max(smoothed)
            )
            
            peaks_info = []
            for i, peak_idx in enumerate(peaks):
                if peak_idx >= len(two_theta):
                    continue
                    
                peak_2theta = two_theta[peak_idx]
                peak_intensity = intensity[peak_idx]
                
                # Estimate FWHM
                if 'widths' in properties and i < len(properties['widths']):
                    width_points = properties['widths'][i]
                    fwhm_estimate = width_points * step_size
                else:
                    fwhm_estimate = 0.1
                
                # Simple peak fitting for FWHM refinement
                fwhm_refined = PeakDetector._estimate_fwhm(two_theta, intensity, peak_2theta)
                
                peaks_info.append({
                    'index': peak_idx,
                    '2theta': peak_2theta,
                    'intensity': peak_intensity,
                    'FWHM_obs': fwhm_refined,
                    'FWHM_inst': 0.05,  # Default instrumental broadening
                    'r_squared': 0.85   # Default quality estimate
                })
            
            return peaks_info
            
        except Exception as e:
            logger.error(f"Peak detection failed: {e}")
            return []
    
    @staticmethod
    def _estimate_fwhm(two_theta: np.ndarray, intensity: np.ndarray, peak_center: float) -> float:
        """Estimate FWHM by finding half-maximum points"""
        try:
            # Find peak region
            mask = np.abs(two_theta - peak_center) <= 1.0
            x_region = two_theta[mask]
            y_region = intensity[mask]
            
            if len(x_region) < 5:
                return 0.1
            
            # Find peak maximum
            max_idx = np.argmax(y_region)
            peak_height = y_region[max_idx]
            baseline = np.min(y_region)
            half_max = baseline + (peak_height - baseline) / 2
            
            # Find half-maximum points
            left_idx = max_idx
            right_idx = max_idx
            
            while left_idx > 0 and y_region[left_idx] > half_max:
                left_idx -= 1
            while right_idx < len(y_region) - 1 and y_region[right_idx] > half_max:
                right_idx += 1
            
            if right_idx > left_idx:
                fwhm = x_region[right_idx] - x_region[left_idx]
                return max(0.05, min(fwhm, 2.0))  # Reasonable bounds
            else:
                return 0.1
                
        except Exception:
            return 0.1

# =============================================================================
# ANALYSIS METHODS
# =============================================================================

class ScherrerAnalyzer:
    """Scherrer analysis implementation"""
    
    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm
    
    def analyze(self, peaks_data: List[Dict], k_constant: float = 0.94) -> Dict:
        """Standard Scherrer analysis with resolution limit detection"""
        if not peaks_data:
            return {'success': False, 'message': 'No peaks provided'}
        
        results = []
        resolution_limited_peaks = 0
        
        for peak in peaks_data:
            try:
                two_theta = peak['2theta']
                fwhm_obs = peak['FWHM_obs']
                fwhm_inst = peak.get('FWHM_inst', 0.05)
                
                # Check if peak is resolution-limited (sample width < 20% of instrumental)
                broadening_ratio = fwhm_obs / fwhm_inst
                
                if broadening_ratio < 1.2:
                    # Peak is resolution-limited - cannot reliably measure crystallite size
                    resolution_limited_peaks += 1
                    # Calculate lower bound (assuming instrumental broadening dominates)
                    beta_rad = np.deg2rad(fwhm_inst)
                    theta_rad = np.deg2rad(two_theta / 2.0)
                    lower_bound_nm = k_constant * self.wavelength / (beta_rad * np.cos(theta_rad))
                    
                    results.append({
                        'two_theta': two_theta,
                        'size_nm': lower_bound_nm,
                        'fwhm_obs': fwhm_obs,
                        'fwhm_corrected': fwhm_inst * 0.5,  # Nominal value
                        'resolution_limited': True,
                        'broadening_ratio': broadening_ratio
                    })
                else:
                    # Normal analysis with deconvolution
                    beta_squared = fwhm_obs**2 - fwhm_inst**2
                    
                    # Safety check for numerical issues
                    if beta_squared < (fwhm_inst * 0.1)**2:
                        beta_corrected = fwhm_inst * 0.1
                    else:
                        beta_corrected = np.sqrt(beta_squared)
                    
                    beta_rad = np.deg2rad(beta_corrected)
                    theta_rad = np.deg2rad(two_theta / 2.0)
                    
                    # Scherrer equation: D = K*lambda/(beta*cos(theta))
                    # wavelength is in nm, beta is in radians, result is in nm
                    crystallite_size_nm = k_constant * self.wavelength / (beta_rad * np.cos(theta_rad))
                    
                    results.append({
                        'two_theta': two_theta,
                        'size_nm': crystallite_size_nm,
                        'fwhm_obs': fwhm_obs,
                        'fwhm_corrected': beta_corrected,
                        'resolution_limited': False,
                        'broadening_ratio': broadening_ratio
                    })
                
            except Exception as e:
                logger.warning(f"Scherrer calculation failed: {e}")
                continue
        
        if not results:
            return {'success': False, 'message': 'No valid peaks for analysis'}
        
        # Separate resolution-limited and measurable sizes
        measurable_sizes = [r['size_nm'] for r in results if not r.get('resolution_limited', False)]
        all_sizes = [r['size_nm'] for r in results]
        
        if len(measurable_sizes) == 0:
            # All peaks are resolution-limited
            return {
                'success': True,
                'method': 'Scherrer',
                'results': results,
                'average_size': np.mean(all_sizes),
                'std_size': np.std(all_sizes),
                'num_peaks': len(results),
                'k_constant': k_constant,
                'resolution_limited': True,
                'warning': f'All {len(results)} peaks are resolution-limited. Crystallites are larger than instrumental resolution can measure. Reported values are lower bounds.'
            }
        elif resolution_limited_peaks > 0:
            return {
                'success': True,
                'method': 'Scherrer',
                'results': results,
                'average_size': np.mean(measurable_sizes),
                'std_size': np.std(measurable_sizes),
                'num_peaks': len(results),
                'num_measurable': len(measurable_sizes),
                'num_resolution_limited': resolution_limited_peaks,
                'k_constant': k_constant,
                'warning': f'{resolution_limited_peaks}/{len(results)} peaks are resolution-limited (sharp peaks with minimal sample broadening)'
            }
        else:
            return {
                'success': True,
                'method': 'Scherrer',
                'results': results,
                'average_size': np.mean(all_sizes),
                'std_size': np.std(all_sizes),
                'num_peaks': len(results),
                'k_constant': k_constant
            }

class MonshiScherrerAnalyzer:
    """Monshi-Scherrer analysis with Ka2 correction"""
    
    def __init__(self, wavelength_info: Dict):
        self.wavelength_info = wavelength_info
        self.ka1 = wavelength_info.get('ka1', 0.15406)
        self.ka2 = wavelength_info.get('ka2', 0.154439)
        self.alpha = wavelength_info.get('alpha', 2.2e-3)
    
    def analyze(self, peaks_data: List[Dict], k_constant: float = 0.94) -> Dict:
        """Monshi-Scherrer analysis with Ka2 correction and resolution limit detection"""
        if not peaks_data:
            return {'success': False, 'message': 'No peaks provided'}
        
        results = []
        resolution_limited_peaks = 0
        
        for peak in peaks_data:
            try:
                two_theta = peak['2theta']
                fwhm_obs = peak['FWHM_obs']
                fwhm_inst = peak.get('FWHM_inst', 0.05)
                
                # Check if peak is resolution-limited
                broadening_ratio = fwhm_obs / fwhm_inst
                
                if broadening_ratio < 1.2:
                    # Resolution-limited peak
                    resolution_limited_peaks += 1
                    beta_rad = np.deg2rad(fwhm_inst * 0.5)
                    theta_rad = np.deg2rad(two_theta / 2.0)
                    
                    b_correction = self.alpha * np.tan(theta_rad)
                    beta_monshi = beta_rad - np.deg2rad(b_correction)
                    beta_monshi = max(beta_monshi, beta_rad * 0.1)
                    
                    lower_bound_nm = k_constant * self.ka1 / (beta_monshi * np.cos(theta_rad))
                    
                    results.append({
                        'two_theta': two_theta,
                        'size_nm': lower_bound_nm,
                        'fwhm_obs': fwhm_obs,
                        'fwhm_corrected': fwhm_inst * 0.5,
                        'fwhm_monshi': np.rad2deg(beta_monshi),
                        'b_correction': b_correction,
                        'resolution_limited': True,
                        'broadening_ratio': broadening_ratio
                    })
                else:
                    # Normal deconvolution
                    beta_squared = fwhm_obs**2 - fwhm_inst**2
                    if beta_squared < (fwhm_inst * 0.1)**2:
                        beta_corrected = fwhm_inst * 0.1
                    else:
                        beta_corrected = np.sqrt(beta_squared)
                    
                    beta_rad = np.deg2rad(beta_corrected)
                    theta_rad = np.deg2rad(two_theta / 2.0)
                    
                    # Monshi-Scherrer correction factor
                    b_correction = self.alpha * np.tan(theta_rad)
                    beta_monshi = beta_rad - np.deg2rad(b_correction)
                    beta_monshi = max(beta_monshi, beta_rad * 0.1)
                    
                    # Modified Scherrer equation (wavelength in nm, result in nm)
                    crystallite_size_nm = k_constant * self.ka1 / (beta_monshi * np.cos(theta_rad))
                    
                    results.append({
                        'two_theta': two_theta,
                        'size_nm': crystallite_size_nm,
                        'fwhm_obs': fwhm_obs,
                        'fwhm_corrected': np.rad2deg(beta_corrected),
                        'fwhm_monshi': np.rad2deg(beta_monshi),
                        'b_correction': b_correction,
                        'resolution_limited': False,
                        'broadening_ratio': broadening_ratio
                    })
                
            except Exception as e:
                logger.warning(f"Monshi-Scherrer calculation failed: {e}")
                continue
        
        if not results:
            return {'success': False, 'message': 'No valid peaks for analysis'}
        
        # Separate resolution-limited and measurable sizes
        measurable_sizes = [r['size_nm'] for r in results if not r.get('resolution_limited', False)]
        all_sizes = [r['size_nm'] for r in results]
        
        if len(measurable_sizes) == 0:
            return {
                'success': True,
                'method': 'Monshi-Scherrer',
                'results': results,
                'average_size': np.mean(all_sizes),
                'std_size': np.std(all_sizes),
                'num_peaks': len(results),
                'k_constant': k_constant,
                'ka1_wavelength': self.ka1,
                'alpha_parameter': self.alpha,
                'resolution_limited': True,
                'warning': f'All {len(results)} peaks are resolution-limited. Crystallites are larger than instrumental resolution can measure.'
            }
        elif resolution_limited_peaks > 0:
            return {
                'success': True,
                'method': 'Monshi-Scherrer',
                'results': results,
                'average_size': np.mean(measurable_sizes),
                'std_size': np.std(measurable_sizes),
                'num_peaks': len(results),
                'num_measurable': len(measurable_sizes),
                'num_resolution_limited': resolution_limited_peaks,
                'k_constant': k_constant,
                'ka1_wavelength': self.ka1,
                'alpha_parameter': self.alpha,
                'warning': f'{resolution_limited_peaks}/{len(results)} peaks are resolution-limited'
            }
        else:
            return {
                'success': True,
                'method': 'Monshi-Scherrer',
                'results': results,
                'average_size': np.mean(all_sizes),
                'std_size': np.std(all_sizes),
                'num_peaks': len(results),
                'k_constant': k_constant,
                'ka1_wavelength': self.ka1,
                'alpha_parameter': self.alpha
            }

class WilliamsonHallAnalyzer:
    """Williamson-Hall analysis implementation"""
    
    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm
    
    def analyze(self, peaks_data: List[Dict]) -> Dict:
        """Williamson-Hall analysis"""
        if len(peaks_data) < 3:
            return {'success': False, 'message': 'Need at least 3 peaks for W-H analysis'}
        
        x_data = []  # 4 sin theta / lambda
        y_data = []  # beta cos theta / lambda
        
        for peak in peaks_data:
            try:
                two_theta = peak['2theta']
                fwhm_obs = peak['FWHM_obs']
                fwhm_inst = peak.get('FWHM_inst', 0.05)
                
                beta_corrected = np.sqrt(max(fwhm_obs**2 - fwhm_inst**2, (fwhm_obs*0.1)**2))
                beta_rad = np.deg2rad(beta_corrected)
                theta_rad = np.deg2rad(two_theta / 2.0)
                
                x_val = 4 * np.sin(theta_rad) / self.wavelength
                y_val = beta_rad * np.cos(theta_rad) / self.wavelength
                
                x_data.append(x_val)
                y_data.append(y_val)
                
            except Exception as e:
                logger.warning(f"W-H calculation failed: {e}")
                continue
        
        if len(x_data) < 3:
            return {'success': False, 'message': 'Insufficient valid data points'}
        
        x_data = np.array(x_data)
        y_data = np.array(y_data)
        
        try:
            # Linear regression
            coeffs = np.polyfit(x_data, y_data, 1)
            slope, intercept = coeffs
            
            # Calculate R-squared
            y_pred = slope * x_data + intercept
            ss_res = np.sum((y_data - y_pred) ** 2)
            ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            
            # Extract physical parameters
            # intercept = K/D in nm^-1, so D = K/intercept in nm
            crystallite_size_nm = 0.94 / intercept if intercept > 0 else float('inf')
            microstrain_percent = (slope / 4) * 100
            
            return {
                'success': True,
                'method': 'Williamson-Hall',
                'crystallite_size_nm': crystallite_size_nm,
                'microstrain_percent': microstrain_percent,
                'r_squared': r_squared,
                'x_data': x_data,
                'y_data': y_data,
                'slope': slope,
                'intercept': intercept,
                'num_peaks': len(x_data)
            }
            
        except Exception as e:
            logger.error(f"W-H linear regression failed: {e}")
            return {'success': False, 'message': f'Linear regression failed: {str(e)}'}

class HalderWagnerAnalyzer:
    """Halder-Wagner analysis implementation"""
    
    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm
    
    def analyze(self, peaks_data: List[Dict], a_parameter: float = 2.0) -> Dict:
        """Halder-Wagner analysis (modified Williamson-Hall)"""
        if len(peaks_data) < 3:
            return {'success': False, 'message': 'Need at least 3 peaks for H-W analysis'}
        
        x_data = []  # (4 sin theta / lambda)^2
        y_data = []  # (beta cos theta / lambda)^2
        
        for peak in peaks_data:
            try:
                two_theta = peak['2theta']
                fwhm_obs = peak['FWHM_obs']
                fwhm_inst = peak.get('FWHM_inst', 0.05)
                
                beta_corrected = np.sqrt(max(fwhm_obs**2 - fwhm_inst**2, (fwhm_obs*0.1)**2))
                beta_rad = np.deg2rad(beta_corrected)
                theta_rad = np.deg2rad(two_theta / 2.0)
                
                # Halder-Wagner coordinates: (4sin(theta)/lambda)^2 vs (beta*cos(theta)/lambda)^2
                x_val = (4 * np.sin(theta_rad) / self.wavelength) ** 2
                y_val = (beta_rad * np.cos(theta_rad) / self.wavelength) ** 2
                
                x_data.append(x_val)
                y_data.append(y_val)
                
            except Exception as e:
                logger.warning(f"H-W calculation failed: {e}")
                continue
        
        if len(x_data) < 3:
            return {'success': False, 'message': 'Insufficient valid data points'}
        
        x_data = np.array(x_data)
        y_data = np.array(y_data)
        
        try:
            # Linear regression: y = slope * x + intercept
            # (beta*cos(theta)/lambda)^2 = (K/D)^2 + (a*strain^2) * (4sin(theta)/lambda)^2
            coeffs = np.polyfit(x_data, y_data, 1)
            slope, intercept = coeffs
            
            # Calculate R-squared
            y_pred = slope * x_data + intercept
            ss_res = np.sum((y_data - y_pred) ** 2)
            ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            
            # Extract physical parameters
            # Intercept = (K/D)^2 in nm^-2 → D = K/sqrt(intercept) in nm
            if intercept > 0:
                crystallite_size_nm = 0.94 / np.sqrt(intercept)
            else:
                crystallite_size_nm = float('inf')
            
            # Slope = a*strain^2 → strain = sqrt(slope/a)
            if slope > 0:
                microstrain_percent = np.sqrt(slope / a_parameter) * 100
            else:
                microstrain_percent = 0
            
            return {
                'success': True,
                'method': 'Halder-Wagner',
                'crystallite_size_nm': crystallite_size_nm,
                'microstrain_percent': microstrain_percent,
                'r_squared': r_squared,
                'x_data': x_data,
                'y_data': y_data,
                'slope': slope,
                'intercept': intercept,
                'num_peaks': len(x_data),
                'a_parameter': a_parameter
            }
            
        except Exception as e:
            logger.error(f"Halder-Wagner linear regression failed: {e}")
            return {'success': False, 'message': f'H-W regression failed: {str(e)}'}

# =============================================================================
# SIMPLIFIED GUI
# =============================================================================

class VolvaGUI:
    """Main GUI application"""
    
    def __init__(self):
        self.config = VolvaConfig()
        
        # Initialize GUI
        self.root = tk.Tk()
        self.root.title("Volva - XRD Analysis Tool v3.0")
        self.root.geometry("1400x900")
        
        # Data storage
        self.sample_data = {'two_theta': None, 'intensity': None, 'filename': None}
        self.lab6_data = {'two_theta': None, 'intensity': None, 'filename': None, 'peaks': None}
        self.instrumental_broadening = {}  # Dict: {2theta: FWHM_inst}
        self.peaks = []
        self.analysis_results = {}
        self.current_wavelength = Constants.WAVELENGTHS["Cu Ka"]["ka1"]
        
        # Analysis components
        self.current_wavelength_info = Constants.WAVELENGTHS["Cu Ka"]
        self.scherrer_analyzer = ScherrerAnalyzer(self.current_wavelength)
        self.monshi_analyzer = MonshiScherrerAnalyzer(self.current_wavelength_info)
        self.wh_analyzer = WilliamsonHallAnalyzer(self.current_wavelength)
        self.hw_analyzer = HalderWagnerAnalyzer(self.current_wavelength)
        
        self.setup_gui()
    
    def setup_gui(self):
        """Setup the main GUI layout"""
        # Main container
        main_paned = ttk.PanedWindow(self.root, orient=tk.HORIZONTAL)
        main_paned.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Left panel for controls
        left_panel = ttk.Frame(main_paned, width=300)
        main_paned.add(left_panel, weight=0)
        
        # Right panel for plot and results
        right_paned = ttk.PanedWindow(main_paned, orient=tk.VERTICAL)
        main_paned.add(right_paned, weight=1)
        
        plot_panel = ttk.Frame(right_paned)
        right_paned.add(plot_panel, weight=2)
        
        results_panel = ttk.Frame(right_paned)
        right_paned.add(results_panel, weight=1)
        
        self.setup_controls(left_panel)
        self.setup_plot(plot_panel)
        self.setup_results(results_panel)
        
        # Status bar
        self.status_var = tk.StringVar(value="Volva Ready - Load XRD data to begin")
        status_bar = ttk.Label(self.root, textvariable=self.status_var, relief=tk.SUNKEN)
        status_bar.pack(side=tk.BOTTOM, fill=tk.X)
    
    def setup_controls(self, parent):
        """Setup control panel"""
        # File loading
        file_frame = ttk.LabelFrame(parent, text="Data Loading", padding=10)
        file_frame.pack(fill=tk.X, pady=5, padx=5)
        
        ttk.Button(file_frame, text="Load XRD Data", command=self.load_data).pack(fill=tk.X, pady=2)
        self.file_label = ttk.Label(file_frame, text="No file loaded", foreground="gray")
        self.file_label.pack(fill=tk.X, pady=2)
        
        ttk.Separator(file_frame, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=5)
        ttk.Button(file_frame, text="Load LaB₆ Standard", command=self.load_lab6_standard).pack(fill=tk.X, pady=2)
        self.lab6_label = ttk.Label(file_frame, text="No LaB₆ standard loaded", foreground="gray")
        self.lab6_label.pack(fill=tk.X, pady=2)
        
        # Wavelength selection
        wave_frame = ttk.LabelFrame(parent, text="X-ray Source", padding=10)
        wave_frame.pack(fill=tk.X, pady=5, padx=5)
        
        self.wavelength_var = tk.StringVar(value="Cu Ka")
        wavelength_combo = ttk.Combobox(wave_frame, textvariable=self.wavelength_var,
                                       values=list(Constants.WAVELENGTHS.keys()), state="readonly")
        wavelength_combo.pack(fill=tk.X)
        wavelength_combo.bind('<<ComboboxSelected>>', self.update_wavelength)
        
        # Peak detection
        peak_frame = ttk.LabelFrame(parent, text="Peak Detection", padding=10)
        peak_frame.pack(fill=tk.X, pady=5, padx=5)
        
        self.prominence_var = tk.DoubleVar(value=0.1)
        ttk.Label(peak_frame, text="Prominence Factor:").pack(anchor=tk.W)
        ttk.Scale(peak_frame, from_=0.01, to=0.5, variable=self.prominence_var, 
                 orient=tk.HORIZONTAL).pack(fill=tk.X)
        
        ttk.Button(peak_frame, text="Detect Peaks", command=self.detect_peaks).pack(fill=tk.X, pady=3)
        self.peaks_label = ttk.Label(peak_frame, text="No peaks detected", foreground="gray")
        self.peaks_label.pack(fill=tk.X, pady=2)
        
        # Analysis methods
        methods_frame = ttk.LabelFrame(parent, text="Analysis Methods", padding=10)
        methods_frame.pack(fill=tk.X, pady=5, padx=5)
        
        ttk.Button(methods_frame, text="Scherrer Analysis", command=self.run_scherrer).pack(fill=tk.X, pady=1)
        ttk.Button(methods_frame, text="Monshi-Scherrer", command=self.run_monshi_scherrer).pack(fill=tk.X, pady=1)
        ttk.Button(methods_frame, text="Williamson-Hall", command=self.run_williamson_hall).pack(fill=tk.X, pady=1)
        ttk.Button(methods_frame, text="Halder-Wagner", command=self.run_halder_wagner).pack(fill=tk.X, pady=1)
        ttk.Separator(methods_frame, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=5)
        ttk.Button(methods_frame, text="Run All Methods", command=self.run_all_methods).pack(fill=tk.X, pady=2)
    
    def setup_plot(self, parent):
        """Setup matplotlib plot"""
        self.fig, self.ax = plt.subplots(figsize=(10, 6))
        self.canvas = FigureCanvasTkAgg(self.fig, parent)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        
        toolbar = NavigationToolbar2Tk(self.canvas, parent)
        toolbar.update()
        
        self.plot_empty()
    
    def setup_results(self, parent):
        """Setup results panel"""
        self.results_text = scrolledtext.ScrolledText(parent, height=10, wrap=tk.WORD, 
                                                     font=('Consolas', 10))
        self.results_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
    
    # GUI Methods
    def load_data(self):
        """Load XRD data file"""
        file_path = filedialog.askopenfilename(
            title="Select XRD Data File",
            filetypes=[("Text files", "*.txt"), ("Data files", "*.dat"), 
                      ("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if file_path:
            two_theta, intensity, message = XRDDataProcessor.load_xrd_file(file_path)
            
            if two_theta is not None:
                self.sample_data = {
                    'two_theta': two_theta,
                    'intensity': intensity,
                    'filename': os.path.basename(file_path)
                }
                self.file_label.config(text=f"✓ {os.path.basename(file_path)}", foreground="darkgreen")
                self.plot_data()
                self.status_var.set(message)
                
                # Reset analysis
                self.peaks = []
                self.analysis_results = {}
            else:
                messagebox.showerror("Error", message)
    
    def load_lab6_standard(self):
        """Load LaB6 standard for instrumental broadening correction"""
        file_path = filedialog.askopenfilename(
            title="Select LaB₆ Standard Data File",
            filetypes=[("Text files", "*.txt"), ("Data files", "*.dat"), 
                      ("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if file_path:
            two_theta, intensity, message = XRDDataProcessor.load_xrd_file(file_path)
            
            if two_theta is not None:
                self.lab6_data = {
                    'two_theta': two_theta,
                    'intensity': intensity,
                    'filename': os.path.basename(file_path),
                    'peaks': None
                }
                
                # Automatically detect peaks in LaB6 data
                try:
                    processed_intensity, _ = XRDDataProcessor.subtract_background(two_theta, intensity)
                    lab6_peaks = PeakDetector.detect_peaks(two_theta, processed_intensity, 
                                                          prominence_factor=0.15, min_width_deg=0.05)
                    
                    if lab6_peaks:
                        self.lab6_data['peaks'] = lab6_peaks
                        
                        # Create instrumental broadening lookup
                        self.instrumental_broadening = {peak['2theta']: peak['FWHM_obs'] 
                                                       for peak in lab6_peaks}
                        
                        self.lab6_label.config(
                            text=f"✓ {os.path.basename(file_path)} ({len(lab6_peaks)} peaks)", 
                            foreground="darkgreen")
                        self.status_var.set(f"LaB₆ standard loaded: {len(lab6_peaks)} reference peaks for instrumental broadening")
                        
                        messagebox.showinfo("LaB₆ Standard Loaded", 
                                          f"Detected {len(lab6_peaks)} peaks in LaB₆ standard.\n"
                                          f"Instrumental broadening will be applied to sample analysis.")
                    else:
                        messagebox.showwarning("Warning", 
                                             "No peaks detected in LaB₆ data. Try adjusting prominence.")
                        self.lab6_label.config(text="⚠ LaB₆ loaded but no peaks detected", foreground="orange")
                        
                except Exception as e:
                    messagebox.showerror("Error", f"Failed to analyze LaB₆ data: {str(e)}")
                    self.lab6_label.config(text="⚠ LaB₆ loaded with errors", foreground="orange")
            else:
                messagebox.showerror("Error", message)
    
    def get_instrumental_broadening(self, two_theta_value: float) -> float:
        """Get instrumental broadening at a given 2theta value via interpolation"""
        if not self.instrumental_broadening:
            return 0.05  # Default if no LaB6 data
        
        two_theta_ref = np.array(list(self.instrumental_broadening.keys()))
        fwhm_inst = np.array(list(self.instrumental_broadening.values()))
        
        # Linear interpolation
        if two_theta_value <= two_theta_ref.min():
            return fwhm_inst[np.argmin(two_theta_ref)]
        elif two_theta_value >= two_theta_ref.max():
            return fwhm_inst[np.argmax(two_theta_ref)]
        else:
            return np.interp(two_theta_value, two_theta_ref, fwhm_inst)
    
    def detect_peaks(self):
        """Detect peaks in loaded data"""
        if self.sample_data['two_theta'] is None:
            messagebox.showwarning("Warning", "Please load XRD data first")
            return
        
        try:
            self.status_var.set("Detecting peaks...")
            self.root.update()
            
            # Background subtraction
            processed_intensity, background = XRDDataProcessor.subtract_background(
                self.sample_data['two_theta'], self.sample_data['intensity'])
            
            # Peak detection
            self.peaks = PeakDetector.detect_peaks(
                self.sample_data['two_theta'], processed_intensity,
                prominence_factor=self.prominence_var.get())
            
            # Apply instrumental broadening from LaB6 data if available
            if self.peaks and self.instrumental_broadening:
                for peak in self.peaks:
                    peak['FWHM_inst'] = self.get_instrumental_broadening(peak['2theta'])
                self.status_var.set(f"Peak detection complete: {len(self.peaks)} peaks (with LaB₆ correction)")
            elif self.peaks:
                self.status_var.set(f"Peak detection complete: {len(self.peaks)} peaks (using default instrumental broadening)")
            
            if self.peaks:
                self.peaks_label.config(text=f"✓ {len(self.peaks)} peaks detected", foreground="darkgreen")
                self.plot_data_with_peaks()
            else:
                messagebox.showinfo("Peak Detection", "No peaks detected. Try adjusting parameters.")
                
        except Exception as e:
            messagebox.showerror("Error", f"Peak detection failed: {str(e)}")
    
    def run_scherrer(self):
        """Run Scherrer analysis"""
        if not self.peaks:
            messagebox.showwarning("Warning", "Please detect peaks first")
            return
        
        try:
            result = self.scherrer_analyzer.analyze(self.peaks)
            if result['success']:
                self.analysis_results['scherrer'] = result
                self.display_results("Scherrer Analysis", result)
            else:
                messagebox.showerror("Analysis Error", result['message'])
        except Exception as e:
            messagebox.showerror("Error", f"Scherrer analysis failed: {str(e)}")
    
    def run_monshi_scherrer(self):
        """Run Monshi-Scherrer analysis"""
        if not self.peaks:
            messagebox.showwarning("Warning", "Please detect peaks first")
            return
        
        try:
            result = self.monshi_analyzer.analyze(self.peaks)
            if result['success']:
                self.analysis_results['monshi_scherrer'] = result
                self.display_results("Monshi-Scherrer Analysis", result)
            else:
                messagebox.showerror("Analysis Error", result['message'])
        except Exception as e:
            messagebox.showerror("Error", f"Monshi-Scherrer analysis failed: {str(e)}")
    
    def run_halder_wagner(self):
        """Run Halder-Wagner analysis"""
        if len(self.peaks) < 3:
            messagebox.showwarning("Warning", f"Need at least 3 peaks for H-W analysis. Found {len(self.peaks)}")
            return
        
        try:
            result = self.hw_analyzer.analyze(self.peaks)
            if result['success']:
                self.analysis_results['halder_wagner'] = result
                self.display_results("Halder-Wagner Analysis", result)
                self.show_hw_plot(result)
            else:
                messagebox.showerror("Analysis Error", result['message'])
        except Exception as e:
            messagebox.showerror("Error", f"Halder-Wagner analysis failed: {str(e)}")
    
    def run_williamson_hall(self):
        """Run Williamson-Hall analysis"""
        if len(self.peaks) < 3:
            messagebox.showwarning("Warning", f"Need at least 3 peaks for W-H analysis. Found {len(self.peaks)}")
            return
        
        try:
            result = self.wh_analyzer.analyze(self.peaks)
            if result['success']:
                self.analysis_results['williamson_hall'] = result
                self.display_results("Williamson-Hall Analysis", result)
                self.show_wh_plot(result)
            else:
                messagebox.showerror("Analysis Error", result['message'])
        except Exception as e:
            messagebox.showerror("Error", f"W-H analysis failed: {str(e)}")
    
    def show_hw_plot(self, result):
        """Show Halder-Wagner plot in new window"""
        try:
            plot_window = tk.Toplevel(self.root)
            plot_window.title("Halder-Wagner Analysis")
            plot_window.geometry("800x600")
            
            fig, ax = plt.subplots(figsize=(8, 6))
            
            x_data = result['x_data']
            y_data = result['y_data']
            
            # Scatter plot
            ax.scatter(x_data, y_data, s=100, alpha=0.7, color='purple', edgecolors='black')
            
            # Fit line
            x_fit = np.linspace(x_data.min(), x_data.max(), 100)
            y_fit = result['slope'] * x_fit + result['intercept']
            ax.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'R² = {result["r_squared"]:.4f}')
            
            # Labels and formatting
            ax.set_xlabel('(4 sin θ / λ)² (nm⁻²)', fontsize=12)
            ax.set_ylabel('(β cos θ / λ)² (nm⁻²)', fontsize=12)
            ax.set_title('Halder-Wagner Analysis', fontsize=14)
            ax.grid(True, alpha=0.3)
            ax.legend()
            
            # Results text
            result_text = f"Size: {result['crystallite_size_nm']:.1f} nm\n"
            result_text += f"Strain: {result['microstrain_percent']:.4f} %"
            ax.text(0.05, 0.95, result_text, transform=ax.transAxes,
                   bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9),
                   va='top', fontsize=11)
            
            canvas = FigureCanvasTkAgg(fig, plot_window)
            canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
            canvas.draw()
            
        except Exception as e:
            messagebox.showerror("Error", f"Failed to create H-W plot: {str(e)}")
    
    def run_all_methods(self):
        """Run all available analysis methods"""
        if not self.peaks:
            messagebox.showwarning("Warning", "Please detect peaks first")
            return
        
        self.results_text.delete(1.0, tk.END)
        self.analysis_results = {}
        
        # Run Scherrer
        scherrer_result = self.scherrer_analyzer.analyze(self.peaks)
        if scherrer_result['success']:
            self.analysis_results['scherrer'] = scherrer_result
            self.display_results("Scherrer Analysis", scherrer_result)
        
        # Run W-H if enough peaks
        if len(self.peaks) >= 3:
            wh_result = self.wh_analyzer.analyze(self.peaks)
            if wh_result['success']:
                self.analysis_results['williamson_hall'] = wh_result
                self.display_results("Williamson-Hall Analysis", wh_result)
        
        self.status_var.set(f"Analysis complete: {len(self.analysis_results)} methods")
    
    def run_all_methods(self):
        """Run all available analysis methods"""
        if not self.peaks:
            messagebox.showwarning("Warning", "Please detect peaks first")
            return
        
        self.results_text.delete(1.0, tk.END)
        self.analysis_results = {}
        
        # Run Scherrer
        scherrer_result = self.scherrer_analyzer.analyze(self.peaks)
        if scherrer_result['success']:
            self.analysis_results['scherrer'] = scherrer_result
            self.display_results("Scherrer Analysis", scherrer_result)
        
        # Run Monshi-Scherrer
        monshi_result = self.monshi_analyzer.analyze(self.peaks)
        if monshi_result['success']:
            self.analysis_results['monshi_scherrer'] = monshi_result
            self.display_results("Monshi-Scherrer Analysis", monshi_result)
        
        # Run W-H if enough peaks
        if len(self.peaks) >= 3:
            wh_result = self.wh_analyzer.analyze(self.peaks)
            if wh_result['success']:
                self.analysis_results['williamson_hall'] = wh_result
                self.display_results("Williamson-Hall Analysis", wh_result)
            
            # Run Halder-Wagner
            hw_result = self.hw_analyzer.analyze(self.peaks)
            if hw_result['success']:
                self.analysis_results['halder_wagner'] = hw_result
                self.display_results("Halder-Wagner Analysis", hw_result)
        
        self.status_var.set(f"Analysis complete: {len(self.analysis_results)} methods")
    
    def update_wavelength(self, event=None):
        """Update wavelength for all analyzers"""
        wavelength_name = self.wavelength_var.get()
        if wavelength_name in Constants.WAVELENGTHS:
            wavelength_data = Constants.WAVELENGTHS[wavelength_name]
            self.current_wavelength = wavelength_data['ka1']
            self.current_wavelength_info = wavelength_data
            
            # Update all analyzers
            self.scherrer_analyzer.wavelength = self.current_wavelength
            self.monshi_analyzer.wavelength_info = self.current_wavelength_info
            self.monshi_analyzer.ka1 = wavelength_data['ka1']
            self.monshi_analyzer.ka2 = wavelength_data['ka2']
            self.monshi_analyzer.alpha = wavelength_data['alpha']
            self.wh_analyzer.wavelength = self.current_wavelength
            self.hw_analyzer.wavelength = self.current_wavelength
            
            self.status_var.set(f"Wavelength updated: {wavelength_name}")
    
    def show_wh_plot(self, result):
        """Show Williamson-Hall plot in new window"""
        try:
            plot_window = tk.Toplevel(self.root)
            plot_window.title("Williamson-Hall Analysis")
            plot_window.geometry("800x600")
            
            fig, ax = plt.subplots(figsize=(8, 6))
            
            x_data = result['x_data']
            y_data = result['y_data']
            
            # Scatter plot
            ax.scatter(x_data, y_data, s=100, alpha=0.7, color='blue', edgecolors='black')
            
            # Fit line
            x_fit = np.linspace(x_data.min(), x_data.max(), 100)
            y_fit = result['slope'] * x_fit + result['intercept']
            ax.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'R² = {result["r_squared"]:.4f}')
            
            # Labels and formatting
            ax.set_xlabel('4 sin θ / λ (nm⁻¹)', fontsize=12)
            ax.set_ylabel('β cos θ / λ (nm⁻¹)', fontsize=12)
            ax.set_title('Williamson-Hall Analysis', fontsize=14)
            ax.grid(True, alpha=0.3)
            ax.legend()
            
            # Results text
            result_text = f"Size: {result['crystallite_size_nm']:.1f} nm\n"
            result_text += f"Strain: {result['microstrain_percent']:.4f} %"
            ax.text(0.05, 0.95, result_text, transform=ax.transAxes,
                   bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9),
                   va='top', fontsize=11)
            
            canvas = FigureCanvasTkAgg(fig, plot_window)
            canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
            canvas.draw()
            
        except Exception as e:
            messagebox.showerror("Error", f"Failed to create W-H plot: {str(e)}")
    
    def plot_empty(self):
        """Plot empty state"""
        self.ax.clear()
        self.ax.set_xlabel('2θ (degrees)', fontsize=12)
        self.ax.set_ylabel('Intensity (counts)', fontsize=12)
        self.ax.set_title('Volva XRD Analysis Tool', fontsize=14, fontweight='bold')
        
        empty_text = ('Load XRD data to begin analysis\n\n'
                     'Available Methods:\n'
                     '• Scherrer Analysis\n'
                     '• Williamson-Hall Analysis')
        
        self.ax.text(0.5, 0.5, empty_text,
                    transform=self.ax.transAxes, ha='center', va='center',
                    fontsize=12, alpha=0.6)
        self.ax.grid(True, alpha=0.3)
        self.canvas.draw()
    
    def plot_data(self):
        """Plot XRD data"""
        if self.sample_data['two_theta'] is None:
            self.plot_empty()
            return
        
        self.ax.clear()
        self.ax.plot(self.sample_data['two_theta'], self.sample_data['intensity'], 
                    'b-', linewidth=1.5, label='XRD Data')
        
        self.ax.set_xlabel('2θ (degrees)', fontsize=12)
        self.ax.set_ylabel('Intensity (counts)', fontsize=12)
        self.ax.set_title(f'XRD Pattern: {self.sample_data["filename"]}', fontsize=13)
        self.ax.grid(True, alpha=0.3)
        self.ax.legend()
        
        self.canvas.draw()
    
    def plot_data_with_peaks(self):
        """Plot XRD data with detected peaks"""
        if self.sample_data['two_theta'] is None:
            return
        
        self.ax.clear()
        
        # Plot data
        self.ax.plot(self.sample_data['two_theta'], self.sample_data['intensity'], 
                    'b-', linewidth=1.5, label='XRD Data', alpha=0.8)
        
        # Plot peaks
        if self.peaks:
            peak_positions = [p['2theta'] for p in self.peaks]
            peak_intensities = [p['intensity'] for p in self.peaks]
            
            self.ax.scatter(peak_positions, peak_intensities, c='red', s=100, 
                          marker='^', zorder=5, alpha=0.8, edgecolors='black',
                          label=f'{len(self.peaks)} Peaks')
            
            # Peak labels
            for i, (pos, intensity) in enumerate(zip(peak_positions, peak_intensities)):
                self.ax.annotate(f'P{i+1}', (pos, intensity),
                               xytext=(5, 10), textcoords='offset points',
                               fontsize=9, fontweight='bold')
        
        self.ax.set_xlabel('2θ (degrees)', fontsize=12)
        self.ax.set_ylabel('Intensity (counts)', fontsize=12)
        self.ax.set_title(f'XRD Analysis: {self.sample_data["filename"]}', fontsize=13)
        self.ax.grid(True, alpha=0.3)
        self.ax.legend()
        
        self.canvas.draw()
    
    def display_results(self, method_name: str, result: Dict):
        """Display analysis results"""
        text = f"\n{'='*50}\n{method_name.upper()}\n{'='*50}\n"
        text += f"Sample: {self.sample_data.get('filename', 'Unknown')}\n"
        text += f"Wavelength: {self.current_wavelength:.5f} nm\n"
        text += f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        
        # Show LaB6 correction status
        if self.instrumental_broadening:
            text += f"✓ Instrumental Broadening: LaB₆ corrected ({len(self.instrumental_broadening)} reference peaks)\n"
        else:
            text += f"⚠ Instrumental Broadening: Default (0.05°) - Load LaB₆ standard for accurate correction\n"
        
        # Show resolution limit warning if present
        if 'warning' in result:
            text += f"⚠ RESOLUTION LIMITED: {result['warning']}\n"
        if result.get('resolution_limited', False):
            text += f"⚠ All peaks are too sharp - crystallites exceed instrumental resolution!\n"
            text += f"   Reported sizes are LOWER BOUNDS (actual sizes are larger)\n"
        
        text += "\n"
        
        if method_name.startswith("Scherrer") or method_name.startswith("Monshi"):
            text += f"Average Crystallite Size: {result['average_size']:.1f} ± {result['std_size']:.1f} nm\n"
            
            # Show peak count details
            if 'num_measurable' in result:
                text += f"Peaks Analyzed: {result['num_peaks']} ({result['num_measurable']} measurable, {result['num_resolution_limited']} resolution-limited)\n"
            else:
                text += f"Peaks Analyzed: {result['num_peaks']}\n"
            
            text += f"K Constant: {result['k_constant']:.2f}\n"
            
            if method_name.startswith("Monshi"):
                text += f"Kα1 Wavelength: {result['ka1_wavelength']:.5f} nm\n"
                text += f"Alpha Parameter: {result['alpha_parameter']:.3e}\n"
            
            text += "\nIndividual Peak Results:\n"
            for i, peak_result in enumerate(result['results']):
                size_str = f"{peak_result['size_nm']:.1f} nm"
                if peak_result.get('resolution_limited', False):
                    size_str = f">{peak_result['size_nm']:.0f} nm (resolution limited)"
                text += f"  Peak {i+1}: {size_str} (2θ = {peak_result['two_theta']:.2f}°)\n"
        
        elif method_name.startswith("Williamson") or method_name.startswith("Halder"):
            text += f"Crystallite Size: {result['crystallite_size_nm']:.1f} nm\n"
            text += f"Microstrain: {result['microstrain_percent']:.4f} %\n"
            text += f"Linear Fit R²: {result['r_squared']:.4f}\n"
            text += f"Peaks Used: {result['num_peaks']}\n"
            
            if method_name.startswith("Halder"):
                text += f"'a' Parameter: {result['a_parameter']:.1f}\n"
        
        self.results_text.insert(tk.END, text + "\n")
        self.results_text.see(tk.END)
    
    def run(self):
        """Start the GUI application"""
        try:
            self.root.mainloop()
        except Exception as e:
            logger.error(f"GUI application error: {e}")
            messagebox.showerror("Application Error", f"An error occurred: {str(e)}")

# =============================================================================
# JUPYTER NOTEBOOK INTERFACE
# =============================================================================

class JupyterVolva:
    """Simplified interface for Jupyter notebooks"""
    
    def __init__(self, wavelength_name: str = "Cu Ka"):
        self.config = VolvaConfig()
        self.wavelength_name = wavelength_name
        self.wavelength = Constants.WAVELENGTHS[wavelength_name]['ka1']
        
        self.scherrer_analyzer = ScherrerAnalyzer(self.wavelength)
        self.wh_analyzer = WilliamsonHallAnalyzer(self.wavelength)
        
        self.data = None
        self.peaks = None
        
        print(f"Volva XRD Analyzer initialized with {wavelength_name} radiation")
        print(f"Wavelength: {self.wavelength:.5f} nm")
    
    def load_data(self, file_path: str):
        """Load XRD data from file"""
        two_theta, intensity, message = XRDDataProcessor.load_xrd_file(file_path)
        
        if two_theta is not None:
            self.data = {'two_theta': two_theta, 'intensity': intensity, 'filename': os.path.basename(file_path)}
            print(f"✓ {message}")
            self.plot_data()
            return True
        else:
            print(f"✗ {message}")
            return False
    
    def detect_peaks(self, prominence_factor: float = 0.1, min_width_deg: float = 0.05, subtract_bg: bool = True):
        """Detect peaks in loaded data"""
        if self.data is None:
            print("Please load data first using load_data()")
            return
        
        try:
            # Background subtraction
            if subtract_bg:
                processed_intensity, background = XRDDataProcessor.subtract_background(
                    self.data['two_theta'], self.data['intensity'])
            else:
                processed_intensity = self.data['intensity']
                background = np.zeros_like(self.data['intensity'])
            
            # Peak detection
            self.peaks = PeakDetector.detect_peaks(
                self.data['two_theta'], processed_intensity, 
                prominence_factor=prominence_factor, min_width_deg=min_width_deg)
            
            if self.peaks:
                print(f"✓ Detected {len(self.peaks)} peaks")
                self.plot_data_with_peaks()
                return self.peaks
            else:
                print("No peaks detected. Try adjusting parameters.")
                return []
                
        except Exception as e:
            print(f"Peak detection failed: {e}")
            return []
    
    def run_scherrer(self, k_constant: float = 0.94):
        """Run Scherrer analysis"""
        if self.peaks is None:
            print("Please detect peaks first using detect_peaks()")
            return None
        
        result = self.scherrer_analyzer.analyze(self.peaks, k_constant)
        if result['success']:
            print(f"Scherrer Analysis Results:")
            print(f"Average Size: {result['average_size']:.1f} ± {result['std_size']:.1f} nm")
            print(f"Peaks: {result['num_peaks']}")
            return result
        else:
            print(f"Scherrer analysis failed: {result['message']}")
            return None
    
class JupyterVolva:
    """Simplified interface for Jupyter notebooks"""
    
    def __init__(self, wavelength_name: str = "Cu Ka"):
        self.config = VolvaConfig()
        self.wavelength_name = wavelength_name
        self.wavelength = Constants.WAVELENGTHS[wavelength_name]['ka1']
        self.wavelength_info = Constants.WAVELENGTHS[wavelength_name]
        
        self.scherrer_analyzer = ScherrerAnalyzer(self.wavelength)
        self.monshi_analyzer = MonshiScherrerAnalyzer(self.wavelength_info)
        self.wh_analyzer = WilliamsonHallAnalyzer(self.wavelength)
        self.hw_analyzer = HalderWagnerAnalyzer(self.wavelength)
        
        self.data = None
        self.peaks = None
        
        print(f"Volva XRD Analyzer initialized with {wavelength_name} radiation")
        print(f"Wavelength: {self.wavelength:.5f} nm")
    
    def load_data(self, file_path: str):
        """Load XRD data from file"""
        two_theta, intensity, message = XRDDataProcessor.load_xrd_file(file_path)
        
        if two_theta is not None:
            self.data = {'two_theta': two_theta, 'intensity': intensity, 'filename': os.path.basename(file_path)}
            print(f"✓ {message}")
            self.plot_data()
            return True
        else:
            print(f"✗ {message}")
            return False
    
    def detect_peaks(self, prominence_factor: float = 0.1, min_width_deg: float = 0.05, subtract_bg: bool = True):
        """Detect peaks in loaded data"""
        if self.data is None:
            print("Please load data first using load_data()")
            return
        
        try:
            # Background subtraction
            if subtract_bg:
                processed_intensity, background = XRDDataProcessor.subtract_background(
                    self.data['two_theta'], self.data['intensity'])
            else:
                processed_intensity = self.data['intensity']
                background = np.zeros_like(self.data['intensity'])
            
            # Peak detection
            self.peaks = PeakDetector.detect_peaks(
                self.data['two_theta'], processed_intensity, 
                prominence_factor=prominence_factor, min_width_deg=min_width_deg)
            
            if self.peaks:
                print(f"✓ Detected {len(self.peaks)} peaks")
                self.plot_data_with_peaks()
                return self.peaks
            else:
                print("No peaks detected. Try adjusting parameters.")
                return []
                
        except Exception as e:
            print(f"Peak detection failed: {e}")
            return []
    
    def run_scherrer(self, k_constant: float = 0.94):
        """Run Scherrer analysis"""
        if self.peaks is None:
            print("Please detect peaks first using detect_peaks()")
            return None
        
        result = self.scherrer_analyzer.analyze(self.peaks, k_constant)
        if result['success']:
            print(f"\nScherrer Analysis Results:")
            print(f"Average Size: {result['average_size']:.1f} ± {result['std_size']:.1f} nm")
            print(f"Peaks: {result['num_peaks']}")
            return result
        else:
            print(f"Scherrer analysis failed: {result['message']}")
            return None
    
    def run_monshi_scherrer(self, k_constant: float = 0.94):
        """Run Monshi-Scherrer analysis"""
        if self.peaks is None:
            print("Please detect peaks first using detect_peaks()")
            return None
        
        result = self.monshi_analyzer.analyze(self.peaks, k_constant)
        if result['success']:
            print(f"\nMonshi-Scherrer Analysis Results:")
            print(f"Average Size: {result['average_size']:.1f} ± {result['std_size']:.1f} nm")
            print(f"Peaks: {result['num_peaks']}")
            print(f"Alpha correction: {result['alpha_parameter']:.3e}")
            return result
        else:
            print(f"Monshi-Scherrer analysis failed: {result['message']}")
            return None
    
    def run_williamson_hall(self):
        """Run Williamson-Hall analysis"""
        if self.peaks is None:
            print("Please detect peaks first using detect_peaks()")
            return None
        
        if len(self.peaks) < 3:
            print(f"Need at least 3 peaks for W-H analysis. Found {len(self.peaks)}")
            return None
        
        result = self.wh_analyzer.analyze(self.peaks)
        if result['success']:
            print(f"\nWilliamson-Hall Analysis Results:")
            print(f"Crystallite Size: {result['crystallite_size_nm']:.1f} nm")
            print(f"Microstrain: {result['microstrain_percent']:.4f} %")
            print(f"R²: {result['r_squared']:.4f}")
            self.plot_williamson_hall(result)
            return result
        else:
            print(f"W-H analysis failed: {result['message']}")
            return None
    
    def run_halder_wagner(self, a_parameter: float = 2.0):
        """Run Halder-Wagner analysis"""
        if self.peaks is None:
            print("Please detect peaks first using detect_peaks()")
            return None
        
        if len(self.peaks) < 3:
            print(f"Need at least 3 peaks for H-W analysis. Found {len(self.peaks)}")
            return None
        
        result = self.hw_analyzer.analyze(self.peaks, a_parameter)
        if result['success']:
            print(f"\nHalder-Wagner Analysis Results:")
            print(f"Crystallite Size: {result['crystallite_size_nm']:.1f} nm")
            print(f"Microstrain: {result['microstrain_percent']:.4f} %")
            print(f"R²: {result['r_squared']:.4f}")
            print(f"'a' parameter: {result['a_parameter']:.1f}")
            self.plot_halder_wagner(result)
            return result
        else:
            print(f"H-W analysis failed: {result['message']}")
            return None
    
    def run_all_methods(self):
        """Run all analysis methods"""
        results = {}
        
        # Scherrer
        scherrer_result = self.run_scherrer()
        if scherrer_result:
            results['scherrer'] = scherrer_result
        
        # Monshi-Scherrer
        monshi_result = self.run_monshi_scherrer()
        if monshi_result:
            results['monshi_scherrer'] = monshi_result
        
        # Williamson-Hall
        wh_result = self.run_williamson_hall()
        if wh_result:
            results['williamson_hall'] = wh_result
        
        # Halder-Wagner
        hw_result = self.run_halder_wagner()
        if hw_result:
            results['halder_wagner'] = hw_result
        
        # Summary comparison
        if len(results) >= 2:
            print(f"\n{'='*60}")
            print("METHOD COMPARISON SUMMARY")
            print(f"{'='*60}")
            
            size_methods = ['scherrer', 'monshi_scherrer']
            strain_methods = ['williamson_hall', 'halder_wagner']
            
            # Size comparison
            size_results = {}
            for method in size_methods:
                if method in results:
                    size_results[method] = results[method]['average_size']
            
            if len(size_results) >= 2:
                sizes = list(size_results.values())
                print(f"Size Methods Agreement:")
                for method, size in size_results.items():
                    print(f"  {method.replace('_', '-').title()}: {size:.1f} nm")
                print(f"  Coefficient of Variation: {(np.std(sizes)/np.mean(sizes)*100):.1f}%")
            
            # Strain comparison
            strain_results = {}
            for method in strain_methods:
                if method in results:
                    strain_results[method] = results[method]['microstrain_percent']
            
            if len(strain_results) >= 2:
                print(f"\nStrain Methods:")
                for method, strain in strain_results.items():
                    size = results[method]['crystallite_size_nm']
                    print(f"  {method.replace('_', '-').title()}: Size = {size:.1f} nm, Strain = {strain:.4f}%")
        
        return results
    
    def plot_data(self):
        """Plot XRD data"""
        if self.data is None:
            return
        
        plt.figure(figsize=(12, 6))
        plt.plot(self.data['two_theta'], self.data['intensity'], 'b-', linewidth=1.5)
        plt.xlabel('2θ (degrees)', fontsize=12)
        plt.ylabel('Intensity (counts)', fontsize=12)
        plt.title(f'XRD Pattern: {self.data["filename"]}', fontsize=13)
        plt.grid(True, alpha=0.3)
        plt.show()
    
    def plot_data_with_peaks(self):
        """Plot XRD data with detected peaks"""
        if self.data is None or self.peaks is None:
            return
        
        plt.figure(figsize=(12, 6))
        plt.plot(self.data['two_theta'], self.data['intensity'], 'b-', linewidth=1.5, 
                label='XRD Data', alpha=0.8)
        
        if self.peaks:
            peak_positions = [p['2theta'] for p in self.peaks]
            peak_intensities = [p['intensity'] for p in self.peaks]
            
            plt.scatter(peak_positions, peak_intensities, c='red', s=100, 
                       marker='^', zorder=5, alpha=0.8, edgecolors='black',
                       label=f'{len(self.peaks)} Peaks')
            
            # Peak labels
            for i, (pos, intensity) in enumerate(zip(peak_positions, peak_intensities)):
                plt.annotate(f'P{i+1}', (pos, intensity),
                           xytext=(5, 10), textcoords='offset points',
                           fontsize=9, fontweight='bold')
        
        plt.xlabel('2θ (degrees)', fontsize=12)
        plt.ylabel('Intensity (counts)', fontsize=12)
        plt.title(f'XRD Analysis: {self.data["filename"]}', fontsize=13)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()
    
    def plot_williamson_hall(self, result):
        """Plot Williamson-Hall analysis"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Main W-H plot
        x_data = result['x_data']
        y_data = result['y_data']
        
        ax1.scatter(x_data, y_data, s=100, alpha=0.7, color='blue', edgecolors='black')
        
        # Fit line
        x_fit = np.linspace(x_data.min(), x_data.max(), 100)
        y_fit = result['slope'] * x_fit + result['intercept']
        ax1.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'R² = {result["r_squared"]:.4f}')
        
        ax1.set_xlabel('4 sin θ / λ (nm⁻¹)', fontsize=12)
        ax1.set_ylabel('β cos θ / λ (nm⁻¹)', fontsize=12)
        ax1.set_title('Williamson-Hall Plot', fontsize=14)
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Results summary
        result_text = f"Crystallite Size: {result['crystallite_size_nm']:.1f} nm\n"
        result_text += f"Microstrain: {result['microstrain_percent']:.4f} %\n"
        result_text += f"R²: {result['r_squared']:.4f}\n"
        result_text += f"Peaks: {result['num_peaks']}"
        
        ax1.text(0.05, 0.95, result_text, transform=ax1.transAxes,
                bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9),
                va='top', fontsize=11, fontweight='bold')
        
        # Residuals plot
        y_pred = result['slope'] * x_data + result['intercept']
        residuals = y_data - y_pred
        
        ax2.scatter(x_data, residuals, s=80, alpha=0.7, color='green', edgecolors='black')
        ax2.axhline(0, color='red', linestyle='--', alpha=0.8)
        ax2.set_xlabel('4 sin θ / λ (nm⁻¹)', fontsize=12)
        ax2.set_ylabel('Residuals', fontsize=12)
        ax2.set_title('Fit Residuals', fontsize=14)
        ax2.grid(True, alpha=0.3)
        
    def plot_halder_wagner(self, result):
        """Plot Halder-Wagner analysis"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Main H-W plot
        x_data = result['x_data']
        y_data = result['y_data']
        
        ax1.scatter(x_data, y_data, s=100, alpha=0.7, color='purple', edgecolors='black')
        
        # Fit line
        x_fit = np.linspace(x_data.min(), x_data.max(), 100)
        y_fit = result['slope'] * x_fit + result['intercept']
        ax1.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'R² = {result["r_squared"]:.4f}')
        
        ax1.set_xlabel('(4 sin θ / λ)² (nm⁻²)', fontsize=12)
        ax1.set_ylabel('(β cos θ / λ)² (nm⁻²)', fontsize=12)
        ax1.set_title('Halder-Wagner Plot', fontsize=14)
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Results summary
        result_text = f"Crystallite Size: {result['crystallite_size_nm']:.1f} nm\n"
        result_text += f"Microstrain: {result['microstrain_percent']:.4f} %\n"
        result_text += f"R²: {result['r_squared']:.4f}\n"
        result_text += f"'a' parameter: {result['a_parameter']:.1f}\n"
        result_text += f"Peaks: {result['num_peaks']}"
        
        ax1.text(0.05, 0.95, result_text, transform=ax1.transAxes,
                bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.9),
                va='top', fontsize=11, fontweight='bold')
        
        # Residuals plot
        y_pred = result['slope'] * x_data + result['intercept']
        residuals = y_data - y_pred
        
        ax2.scatter(x_data, residuals, s=80, alpha=0.7, color='orange', edgecolors='black')
        ax2.axhline(0, color='red', linestyle='--', alpha=0.8)
        ax2.set_xlabel('(4 sin θ / λ)² (nm⁻²)', fontsize=12)
        ax2.set_ylabel('Residuals', fontsize=12)
        ax2.set_title('Fit Residuals', fontsize=14)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

    def plot_method_comparison(self, results):
        """Plot comparison of all methods"""
        if len(results) < 2:
            print("Need at least 2 methods for comparison")
            return
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('Complete Method Comparison', fontsize=16, fontweight='bold')
        
        # Size comparison
        size_methods = []
        sizes = []
        colors = ['skyblue', 'lightcoral', 'lightgreen', 'gold']
        
        method_mapping = {
            'scherrer': ('Scherrer', 'average_size'),
            'monshi_scherrer': ('Monshi-Scherrer', 'average_size'),
            'williamson_hall': ('Williamson-Hall', 'crystallite_size_nm'),
            'halder_wagner': ('Halder-Wagner', 'crystallite_size_nm')
        }
        
        for method_key, (display_name, size_key) in method_mapping.items():
            if method_key in results:
                size_methods.append(display_name)
                sizes.append(results[method_key][size_key])
        
        if size_methods:
            bars = ax1.bar(size_methods, sizes, color=colors[:len(size_methods)], 
                          alpha=0.7, edgecolor='black')
            ax1.set_ylabel('Crystallite Size (nm)')
            ax1.set_title('Size Comparison - All Methods')
            ax1.grid(True, alpha=0.3, axis='y')
            
            # Add value labels
            for bar, size in zip(bars, sizes):
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + max(sizes)*0.02,
                        f'{size:.1f}', ha='center', va='bottom', fontweight='bold')
        
        # Strain comparison
        strain_methods = []
        strains = []
        strain_colors = ['lightgreen', 'gold']
        
        for method_key, display_name in [('williamson_hall', 'W-H'), ('halder_wagner', 'H-W')]:
            if method_key in results:
                strain_methods.append(display_name)
                strains.append(results[method_key]['microstrain_percent'])
        
        if strain_methods:
            bars = ax2.bar(strain_methods, strains, color=strain_colors[:len(strain_methods)], 
                          alpha=0.7, edgecolor='black')
            ax2.set_ylabel('Microstrain (%)')
            ax2.set_title('Strain Comparison')
            ax2.grid(True, alpha=0.3, axis='y')
            
            for bar, strain in zip(bars, strains):
                height = bar.get_height()
                ax2.text(bar.get_x() + bar.get_width()/2., height + max(strains)*0.02,
                        f'{strain:.4f}', ha='center', va='bottom', fontweight='bold')
        
        # Method agreement (size methods only)
        if 'scherrer' in results and 'monshi_scherrer' in results:
            scherrer_size = results['scherrer']['average_size']
            monshi_size = results['monshi_scherrer']['average_size']
            
            ax3.scatter([scherrer_size], [monshi_size], s=150, c='blue', alpha=0.7, edgecolors='black')
            
            # Perfect agreement line
            size_range = [min(scherrer_size, monshi_size) * 0.9, max(scherrer_size, monshi_size) * 1.1]
            ax3.plot(size_range, size_range, 'k--', alpha=0.7, label='Perfect Agreement')
            
            diff_percent = ((monshi_size - scherrer_size) / scherrer_size * 100)
            ax3.text(0.05, 0.95, f'Difference: {diff_percent:+.1f}%',
                    transform=ax3.transAxes, bbox=dict(boxstyle="round", facecolor="white"),
                    va='top', fontweight='bold')
            
            ax3.set_xlabel('Scherrer Size (nm)')
            ax3.set_ylabel('Monshi-Scherrer Size (nm)')
            ax3.set_title('Method Agreement')
            ax3.grid(True, alpha=0.3)
            ax3.legend()
        
        # R² comparison
        methods_r2 = []
        r2_values = []
        
        for method_key, (display_name, _) in method_mapping.items():
            if method_key in results:
                methods_r2.append(display_name)
                if 'r_squared' in results[method_key]:
                    r2_values.append(results[method_key]['r_squared'])
                else:
                    r2_values.append(0.85)  # Default for size-only methods
        
        if methods_r2:
            bars = ax4.bar(methods_r2, r2_values, color=colors[:len(methods_r2)], 
                          alpha=0.7, edgecolor='black')
            ax4.set_ylabel('R² / Quality Score')
            ax4.set_title('Fit Quality Comparison')
            ax4.set_ylim(0, 1)
            ax4.grid(True, alpha=0.3, axis='y')
            
            for bar, r2 in zip(bars, r2_values):
                height = bar.get_height()
                ax4.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                        f'{r2:.3f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()

# =============================================================================
# CONVENIENCE FUNCTIONS
# =============================================================================

def launch_gui():
    """Launch the full GUI application"""
    try:
        app = VolvaGUI()
        app.run()
    except Exception as e:
        print(f"GUI launch failed: {e}")
        print("Make sure tkinter is available in your Python installation")

def create_jupyter_analyzer(wavelength: str = "Cu Ka"):
    """Create a Jupyter-friendly analyzer instance"""
    return JupyterVolva(wavelength)

def quick_analysis(file_path: str, wavelength: str = "Cu Ka"):
    """Quick analysis function for testing"""
    analyzer = JupyterVolva(wavelength)
    
    if analyzer.load_data(file_path):
        peaks = analyzer.detect_peaks()
        if peaks:
            results = analyzer.run_all_methods()
            return analyzer, results
    
    return analyzer, None

# =============================================================================
# EXAMPLE USAGE
# =============================================================================

def demo_with_sample_data():
    """Create demo with synthetic XRD data"""
    print("Creating sample XRD data for demonstration...")
    
    # Generate synthetic XRD pattern with multiple peaks
    two_theta = np.linspace(20, 80, 1000)
    
    # Multiple peaks with different intensities and widths to test all methods
    intensity = (1000 * np.exp(-(two_theta - 28.4)**2 / (2 * 0.1**2)) +  # Strong narrow peak
                800 * np.exp(-(two_theta - 40.5)**2 / (2 * 0.15**2)) +   # Medium peak
                600 * np.exp(-(two_theta - 50.3)**2 / (2 * 0.12**2)) +   # Another medium peak
                500 * np.exp(-(two_theta - 58.3)**2 / (2 * 0.2**2)) +    # Broader peak
                400 * np.exp(-(two_theta - 69.1)**2 / (2 * 0.25**2)) +   # Broadest peak
                np.random.normal(0, 20, 1000) + 100)  # Noise + background
    
    # Save sample data
    sample_file = "sample_xrd_data.txt"
    with open(sample_file, 'w') as f:
        f.write("# Sample XRD data for Volva testing - Contains 5 peaks for comprehensive analysis\n")
        f.write("# 2theta(deg)    Intensity(counts)\n")
        for tt, intens in zip(two_theta, intensity):
            f.write(f"{tt:.4f}    {intens:.1f}\n")
    
    print(f"Sample data saved to: {sample_file}")
    
    # Run comprehensive analysis
    analyzer = JupyterVolva("Cu Ka")
    
    if analyzer.load_data(sample_file):
        print("\nDetecting peaks...")
        peaks = analyzer.detect_peaks(prominence_factor=0.15)
        
        if peaks and len(peaks) >= 3:
            print(f"\nRunning all four analysis methods on {len(peaks)} peaks...")
            results = analyzer.run_all_methods()
            
            if results:
                print(f"\nDemo Analysis Complete! Successfully ran {len(results)}/4 methods")
                print("\nTo run with your own data:")
                print("  analyzer = create_jupyter_analyzer()")
                print("  analyzer.load_data('your_file.txt')")
                print("  analyzer.detect_peaks()")
                print("  analyzer.run_all_methods()")
                print("\nTo launch full GUI: launch_gui()")
                
                # Show method comparison plot
                if len(results) >= 2:
                    analyzer.plot_method_comparison(results)
                
                return analyzer, results
        
    return analyzer, None

# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("Volva XRD Analysis Tool v3.0 - Complete Edition")
    print("="*60)
    print("\nUsage options:")
    print("\n1. Jupyter Notebook (Full Analysis):")
    print("   analyzer = create_jupyter_analyzer('Cu Ka')")
    print("   analyzer.load_data('your_file.txt')")
    print("   analyzer.detect_peaks()")
    print("   results = analyzer.run_all_methods()  # All 4 methods")
    print("   analyzer.plot_method_comparison(results)")
    print()
    print("2. Individual Methods:")
    print("   scherrer_result = analyzer.run_scherrer()")
    print("   monshi_result = analyzer.run_monshi_scherrer()")
    print("   wh_result = analyzer.run_williamson_hall()")
    print("   hw_result = analyzer.run_halder_wagner()")
    print()
    print("3. Launch Full GUI:")
    print("   launch_gui()")
    print()
    print("4. Quick Demo (All 4 Methods):")
    print("   analyzer, results = demo_with_sample_data()")
    print()
    print("Available Analysis Methods:")
    print("• Scherrer Analysis - Basic crystallite size")
    print("• Monshi-Scherrer Analysis - With Kα2 correction")
    print("• Williamson-Hall Analysis - Size + microstrain")
    print("• Halder-Wagner Analysis - Modified W-H method")
    
    # Uncomment the line below to run demo automatically
    # analyzer, results = demo_with_sample_data()


# In[ ]:


launch_gui()


#

In [ ]:
# launch_gui()